In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import pickle
from datetime import datetime

In [2]:
data = pd.read_excel("Data/out-of-sample/dataset_comp_0322.xlsx")

## 2023–mid 2025 backtesting

Extracting from yahoo finance the adjusted price for corporate splits and dividends in the period of next 3 months -> i.e. still dec 2022 composition.

In [3]:
# Define the tickers and date range
tickers = list(data['SYMBOL'].values) # Add more tickers as needed
start_date = "2022-03-31"
end_date = "2022-07-01"

# Download the data
yahoo_data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# Extract adjusted close prices
adj_close_bt = pd.DataFrame()

# If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
if len(tickers) == 1:
    adj_close_bt[tickers[0]] = yahoo_data['Adj Close']
else:
    for ticker in tickers:
        try:
            adj_close_bt[ticker] = yahoo_data[ticker]['Adj Close']
        except KeyError:
            print(f"No data found for {ticker}.")

# Display the result
print(adj_close_bt.head())


18 Failed downloads:
['ATVI', 'WRK', 'ABMD', 'TWTR', 'NLSN', 'DISH', 'CERN', 'DRE', 'CTXS', 'SIVBQ', 'PXD', 'CTLT', 'MRO', 'PBCT']: YFTzMissingError('possibly delisted; no timezone found')
['JNPR', 'DFS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2022-03-31 -> 2022-07-01) (Yahoo error = "No data found, symbol may be delisted")')
['BF.B']: YFPricesMissingError('possibly delisted; no price data found  (1d 2022-03-31 -> 2022-07-01)')
['SBNY']: YFPricesMissingError('possibly delisted; no price data found  (1d 2022-03-31 -> 2022-07-01) (Yahoo error = "Data doesn\'t exist for startDate = 1648699200, endDate = 1656648000")')


                  AMZN         ABT        AES  ABMD         IBM         AMD  \
Date                                                                          
2022-03-31  162.997498  110.687332  22.366329   NaN  112.985992  109.339996   
2022-04-01  163.559998  110.893082  22.792267   NaN  113.098953  108.190002   
2022-04-04  168.346497  111.173630  22.740114   NaN  113.203224  110.529999   
2022-04-05  164.054993  111.734734  22.548874   NaN  112.004028  106.820000   
2022-04-06  158.755997  112.164925  22.149006   NaN  112.438515  103.669998   

                  ADBE         ARE         APD        ALK  ...       WELL  \
Date                                                       ...              
2022-03-31  455.619995  173.829102  230.143921  58.009998  ...  87.800674   
2022-04-01  458.190002  177.905960  229.296707  57.950001  ...  89.334953   
2022-04-04  468.809998  175.427032  232.667191  57.160000  ...  89.197952   
2022-04-05  458.579987  175.116058  231.773911  56.869999  ..

/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/1299046644.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adj_close_bt[ticker] = yahoo_data[ticker]['Adj Close']
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/1299046644.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adj_close_bt[ticker] = yahoo_data[ticker]['Adj Close']
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/1299046644.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

In [4]:
#print(data.loc[data['SYMBOL'] == 'BF.B'])
# Define the tickers and date range
tickers = ['BF-B'] # Add more tickers as needed
start_date = "2022-03-31"
end_date = "2022-07-01"


# Download the data
data_bfb = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# Extract adjusted close prices
adj_close_bfb = pd.DataFrame()

# If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
if len(tickers) == 1:
    adj_close_bfb[tickers[0]] = data_bfb['BF-B']['Adj Close']
else:
    for ticker in tickers:
        try:
            adj_close_bfb[ticker] = data_bfb[ticker]['Adj Close']
        except KeyError:
            print(f"No data found for {ticker}.")

# Display the result
#print(adj_close_bfb.head())
adj_close_bt['BF.B'] = adj_close_bfb.values

In [5]:
price_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='CLOSE PRICE', header = 4)
price_next_3m.columns = price_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
price_next_3m = price_next_3m.iloc[:-1]

div_rate_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='DIV RATE', header = 4)
div_rate_next_3m.columns = div_rate_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_rate_next_3m = div_rate_next_3m.iloc[:-1]

div_date_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='DIV DATE', header = 4)
div_date_next_3m.columns = div_date_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_date_next_3m = div_date_next_3m.iloc[:-1]

ffnosh_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='FFNOSH', header = 4)
ffnosh_next_3m.columns =ffnosh_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
ffnosh_next_3m = ffnosh_next_3m.iloc[:-1]


div_date_next_3m.columns = price_next_3m.columns
div_rate_next_3m.columns = price_next_3m.columns
ffnosh_next_3m.columns = price_next_3m.columns

price_next_3m.index = price_next_3m.Code
div_rate_next_3m.index = div_rate_next_3m.Code
div_date_next_3m.index = div_date_next_3m.Code
ffnosh_next_3m.index = ffnosh_next_3m.Code

price_next_3m = price_next_3m.iloc[:, 1:]
div_rate = div_rate_next_3m.iloc[:, 1:]
div_date_next_3m = div_date_next_3m.iloc[:, 1:]
ffnosh_next_3m = ffnosh_next_3m.iloc[:, 1:]

# --- Step 1: Calculate adjustment factors
adj_factors = pd.DataFrame(1.0, index=price_next_3m.index, columns=price_next_3m.columns)

for company in price_next_3m.columns:
    for i in range(1, len(price_next_3m)):
        date = price_next_3m.index[i]
        prev_date = price_next_3m.index[i - 1]

        # If ex-dividend happens on this day
        if pd.notna(div_date_next_3m.at[date, company]):
            div = div_rate.at[date, company]
            price_prev = price_next_3m.at[prev_date, company]
            if price_prev and price_prev != 0:
                factor = (price_prev - div) / price_prev
                adj_factors.at[date, company] = factor

# --- Step 2: Calculate cumulative adjustment factors in reverse (like Yahoo)
cum_factors = adj_factors.iloc[::-1].cumprod().iloc[::-1]

# --- Step 3: Build adjusted prices
adjusted_prices_calculated = price_next_3m * cum_factors



for stock in ['NLSN', 'MRO', 'DISH', 'CTLT', 'PBCT', 'WRK', 'ABMD', 'CERN', 'CTXS', 'PXD', 'DRE', 'TWTR', 'ATVI', 'SIVBQ','SBNY','DFS', 'JNPR']: 
    type_stock_missing = data.loc[data['SYMBOL'] == stock, 'TYPE'].values
    adj_close_bt[stock] = adjusted_prices_calculated.loc[adj_close_bt.index,  type_stock_missing].values

In [6]:
adj_close_bt.to_excel("Data/out-of-sample/adj_price_yahoo_comp_0322_next_3m.xlsx")

In [7]:
adj_close_bt = pd.read_excel("Data/out-of-sample/adj_price_yahoo_comp_0322_next_3m.xlsx")
adj_close_bt.index = adj_close_bt.Date
adj_close_bt.drop(columns='Date', inplace=True)

In [8]:
nan_columns = adj_close_bt.columns[adj_close_bt.isna().any()]
print(nan_columns)

Index([], dtype='object')


In [9]:
adj_close_bt.isna().any().any()

False

### Market-cap–weighted index (S&P 500 approximation):

Start from Dec 2022 composition, but instead of freezing weights, I recalculate weights each day 

-> This way, weights “evolve naturally with returns” 

-> This mimics how the real S&P 500 works (ignoring reconstitutions)

vs 

 Buy-and-hold portfolio 

You fix the weights at Dec 2022 values (e.g., Apple 7%, Microsoft 6%).

You do not update them after that.

The portfolio return is just the weighted sum of stock returns with those fixed weights.

Over time, it drifts away from being market-cap proportional.

👉 Analogy: If you buy the S&P 500 in Dec 2022 and literally never rebalance, you’re doing buy-and-hold.

In [10]:
data

,Unnamed: 0,NAME,TYPE,SYMBOL,GICS Sector,ffnosh last day Mar 22,Price last day Mar 22,Scope 1,Scope 2,Scope 3,...,Sector_Information Technology,Sector_Materials,Sector_Real Estate,Sector_Utilities,Scope 1 Imputed,Scope 2 Imputed,Scope 3 Imputed,weight_in_sector,float_mcap,rank_in_sector
0,0,AMAZON.COM,891399,AMZN,Consumer Discretionary,8496914,162.9975,13020000.0,3060000.0,49020000.0,...,0,0,0,0,NaN,NaN,NaN,0.327186,1.384976e+09,1
1,1,ABBOTT LABORATORIES,916328,ABT,Health Care,1750942,118.3600,530000.0,476000.0,12829000.0,...,0,0,0,0,NaN,NaN,NaN,0.042589,2.072415e+08,7
2,2,AES,545101,AES,Utilities,543370,25.7300,39931000.0,479000.0,3118000.0,...,0,0,0,1,NaN,NaN,NaN,0.014513,1.398091e+07,23
3,3,ABIOMED DEAD - DELIST.23/12/22,777953,ABMD,Health Care,39106,331.2400,15008000.0,8000000.0,235000000.0,...,0,0,0,0,1.0,1.0,1.0,0.002662,1.295347e+07,58
4,4,INTERNATIONAL BUS.MCHS.,906187,IBM,Information Technology,777622,130.0200,79000.0,330000.0,586000.0,...,1,0,0,0,NaN,NaN,NaN,0.011573,1.011064e+08,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493,493,WEC ENERGY GROUP,902335,WEC,Utilities,293408,99.8100,18865000.0,231000.0,28348000.0,...,0,0,0,1,NaN,NaN,NaN,0.030400,2.928505e+07,11
494,494,MONSTER BEVERAGE,512785,MNST,Consumer Staples,757031,39.9500,10077.0,8300.0,1854377.0,...,0,0,0,0,NaN,NaN,NaN,0.012566,3.024339e+07,21
495,495,LINDE (NYS),9373MH,LIN,Materials,503282,319.4299,16813000.0,20903000.0,25859001.0,...,0,1,0,0,NaN,NaN,NaN,0.172141,1.607633e+08,1
496,496,SBA COMMS.,699786,SBAC,Real Estate,101563,344.1001,11100.0,21000.0,187520.0,...,0,0,1,0,NaN,NaN,1.0,0.035481,3.494784e+07,10


In [11]:
price_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='CLOSE PRICE', header = 4)
price_next_3m.columns = price_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
price_next_3m = price_next_3m.iloc[:-1]

div_rate_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='DIV RATE', header = 4)
div_rate_next_3m.columns = div_rate_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_rate_next_3m = div_rate_next_3m.iloc[:-1]

div_date_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='DIV DATE', header = 4)
div_date_next_3m.columns = div_date_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_date_next_3m = div_date_next_3m.iloc[:-1]

ffnosh_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='FFNOSH', header = 4)
ffnosh_next_3m.columns =ffnosh_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
ffnosh_next_3m = ffnosh_next_3m.iloc[:-1]


div_date_next_3m.columns = price_next_3m.columns
div_rate_next_3m.columns = price_next_3m.columns
ffnosh_next_3m.columns = price_next_3m.columns

price_next_3m.index = price_next_3m.Code
div_rate_next_3m.index = div_rate_next_3m.Code
div_date_next_3m.index = div_date_next_3m.Code
ffnosh_next_3m.index = ffnosh_next_3m.Code

price_next_3m = price_next_3m.iloc[:, 1:]
div_rate = div_rate_next_3m.iloc[:, 1:]
div_date_next_3m = div_date_next_3m.iloc[:, 1:]
ffnosh_next_3m = ffnosh_next_3m.iloc[:, 1:]

type_lseg = pd.read_excel("Data/out-of-sample/symbol_comp_0322.xlsm", sheet_name="SYMBOL", dtype=str).iloc[0].values[1:]
symbol_lseg = pd.read_excel("Data/out-of-sample/symbol_comp_0322.xlsm", sheet_name="SYMBOL", dtype=str, header=2)
symbol_lseg = symbol_lseg.iloc[:1].transpose().reset_index().rename(columns= {"index": "NAME", 0: "SYMBOL"}).iloc[1:]
symbol_lseg['TYPE'] = type_lseg
duplicates = symbol_lseg[symbol_lseg['SYMBOL'].duplicated()]
print(duplicates)
symbol_type_matches  = symbol_lseg.set_index("TYPE")["SYMBOL"].to_dict()

price_next_3m = price_next_3m.rename(columns=symbol_type_matches)
ffnosh_next_3m = ffnosh_next_3m.rename(columns=symbol_type_matches)


# change symbol from BRK.A to BRK-B:
price_next_3m.rename(columns={'BRK.A':'BRK-B'}, inplace=True)
ffnosh_next_3m.rename(columns={'BRK.A':'BRK-B'}, inplace=True)

price_next_3m.index = pd.to_datetime(price_next_3m.index)
ffnosh_next_3m.index = pd.to_datetime(ffnosh_next_3m.index)

# Example duplicate map
duplicate_map = {
    "GOOG": "GOOGL",  # aggregate under GOOG
    "FOX": "FOXA",    # aggregate under FOXA
    "NWS": "NWSA"     # aggregate under NWSA
}

# --- Step 1: Align indexes ---
price_next_3m.index = pd.to_datetime(price_next_3m.index)
ffnosh_next_3m.index = pd.to_datetime(ffnosh_next_3m.index)

# --- Step 2: Apply duplicate mapping ---
price_next_3m = price_next_3m.rename(columns=duplicate_map)
ffnosh_next_3m = ffnosh_next_3m.rename(columns=duplicate_map)

# After renaming, duplicates will exist → group and aggregate
def aggregate_duplicates(price_df, ffnosh_df):
    # Weighted price = Σ(price * shares) / Σ(shares)
    weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
    total_ffnosh = ffnosh_df.groupby(axis=1, level=0).sum()
    return weighted_price, total_ffnosh

price_agg, ffnosh_agg = aggregate_duplicates(price_next_3m, ffnosh_next_3m)



# --- Step 3: Float-adjusted market cap ---
float_mcap = price_agg * ffnosh_agg


# --- Step 4: Add sector info ---
sector_map = data.set_index("SYMBOL")["GICS Sector"].to_dict()
sector_map_series = pd.Series(sector_map)

# Columns in float_mcap but not in sector_map_series
extra_in_float_mcap = float_mcap.columns.difference(sector_map_series.index)

# Symbols in sector_map_series but not in float_mcap
extra_in_sector_map = sector_map_series.index.difference(float_mcap.columns)

print("In float_mcap but not in sector_map_series:", extra_in_float_mcap.tolist())
print("In sector_map_series but not in float_mcap:", extra_in_sector_map.tolist())

# Make sure columns align
#float_mcap = float_mcap.loc[:, sector_map_series.index]

# --- Step 5: Compute sector-level weights daily ---
sector_weights = {}
for sector in sector_map_series.unique():
    tickers = sector_map_series[sector_map_series == sector].index
    sector_mcap = float_mcap[tickers].sum(axis=1)
    sector_weights[sector] = float_mcap[tickers].div(sector_mcap, axis=0)

# Example: weights for Tech sector
print(sector_weights["Information Technology"].head())


                                          NAME SYMBOL    TYPE
370  DISCOVERY SERIES C DEAD - DELIST.11/04/22    WBD  54099E
418                                   NEWS 'B'   NWSA  89257J
422                           UNDER ARMOUR 'C'    UAA  7025QJ
442                                      FOX B   FOXA  9406MA
500                               ALPHABET 'C'  GOOGL  871997
In float_mcap but not in sector_map_series: ['CEG', 'OGN']
In sector_map_series but not in float_mcap: []
                 IBM       AMD      ADBE       ADI      ANSS      AAPL  \
Code                                                                     
2019-01-01  0.027352  0.005659  0.031296  0.009723  0.003480  0.199814   
2019-01-02  0.027729  0.005774  0.031073  0.009735  0.003482  0.200093   
2019-01-03  0.028709  0.005523  0.031530  0.009663  0.003541  0.190327   
2019-01-04  0.028556  0.005892  0.031650  0.009475  0.003538  0.189974   
2019-01-07  0.028508  0.006323  0.031801  0.009452  0.003596  0.187902   

   

/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/1580266900.py:69: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/1580266900.py:69: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/1580266900.py:70: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  total_ffnosh = ffnosh_df.groupby(axis=1, level=0).sum()


In [12]:
for sector, w_df in sector_weights.items():
    assert all(abs(w_df.sum(axis=1) - 1) < 1e-8), f"Sector {sector} not summing to 1"

In [13]:
all_weights = pd.concat(sector_weights.values(), axis=1)
print(all_weights.sum(axis=1).head())

Code
2019-01-01    11.0
2019-01-02    11.0
2019-01-03    11.0
2019-01-04    11.0
2019-01-07    11.0
dtype: float64


In [14]:
daily_returns_next_3m = adj_close_bt.pct_change().dropna()

In [15]:
# Columns in float_mcap but not in sector_map_series
extra_in_float_mcap = float_mcap.columns.difference(daily_returns_next_3m.columns)


print("In float_mcap but not in daily_returns:", extra_in_float_mcap.tolist())



In float_mcap but not in daily_returns: ['CEG', 'OGN']


In [16]:
sector_portfolio_returns = {}

for sector, weights_df in sector_weights.items():
    # Align weights with returns (common tickers + dates)
    common_tickers = weights_df.columns.intersection(daily_returns_next_3m.columns)
    common_index = weights_df.index.intersection(daily_returns_next_3m.index)

    w = weights_df.loc[common_index, common_tickers]
    r = daily_returns_next_3m.loc[common_index, common_tickers]

    # Multiply elementwise and sum across tickers → portfolio return per day
    sector_returns = (w * r).sum(axis=1)

    sector_portfolio_returns[sector] = sector_returns

# Put all in a single DataFrame
sector_portfolio_df = pd.DataFrame(sector_portfolio_returns)
print(sector_portfolio_df.head())


            Consumer Discretionary  Health Care  Utilities  \
2022-04-01                0.002400     0.009212   0.014201   
2022-04-04                0.024692    -0.007388  -0.008007   
2022-04-05               -0.024417     0.002007   0.006827   
2022-04-06               -0.028160     0.015839   0.020168   
2022-04-07                0.002585     0.018821  -0.002730   

            Information Technology  Real Estate  Materials  Industrials  \
2022-04-01               -0.003921     0.020340   0.011956    -0.005611   
2022-04-04                0.020254    -0.002820  -0.000327    -0.001238   
2022-04-05               -0.023175     0.000964  -0.007818    -0.012693   
2022-04-06               -0.025377     0.015602  -0.011001    -0.007482   
2022-04-07                0.004014    -0.008453   0.006178     0.002988   

            Financials    Energy  Communication Services  Consumer Staples  
2022-04-01    0.002278  0.008619                0.009133          0.011399  
2022-04-04   -0.000775

In [17]:
def compute_fixed_weight_returns(returns_df, weights, stock_labels):
    """
    Compute fixed-weight portfolio returns (decarbonised portfolio).
    
    Args:
        returns_df: DataFrame of monthly returns
        weights: np.array of optimised weights
        stock_labels: list or Index of stock tickers

    Returns:
        Series of portfolio returns over time
    """
    sector_returns = returns_df[stock_labels]
    portfolio_returns = sector_returns.dot(weights)
    return portfolio_returns

In [18]:
with open('Data/out-of-sample/optimal_portfolios_0322_minimum_weight.pkl', 'rb') as f:
    optimal_portfolios_shrink = pickle.load(f)

In [19]:
decarb_sector_returns = {}

for sector, data in optimal_portfolios_shrink.items():
    w_opt = data['w_opt']
    tickers = data['stock_labels']
    
    sector_returns = compute_fixed_weight_returns(
        daily_returns_next_3m,
        weights=w_opt,
        stock_labels=tickers
    )
    
    decarb_sector_returns[sector] = sector_returns

In [20]:
benchmark_sectors_daily_returns = sector_portfolio_df
optimised_sectors_daily_returns = pd.DataFrame(decarb_sector_returns)

In [21]:
import numpy as np
import pandas as pd

sector_te = {}

for sector in benchmark_sectors_daily_returns.columns:
    # Align indices
    r_b = benchmark_sectors_daily_returns[sector]
    r_d = optimised_sectors_daily_returns[sector]
    common_idx = r_b.index.intersection(r_d.index)
    r_b, r_d = r_b.loc[common_idx], r_d.loc[common_idx]

    # Active returns
    active = r_d - r_b

    # Daily TE → annualised
    te_daily = active.std()
    te_ann = te_daily * np.sqrt(252)

    sector_te[sector] = te_ann

# Build DataFrame
te_df = pd.DataFrame.from_dict(sector_te, orient="index", columns=["Annualised_TE"])
print(te_df.sort_values("Annualised_TE", ascending=False).round(4))


                        Annualised_TE
Real Estate                    0.0530
Communication Services         0.0466
Consumer Discretionary         0.0371
Industrials                    0.0309
Consumer Staples               0.0291
Health Care                    0.0289
Energy                         0.0276
Materials                      0.0269
Utilities                      0.0258
Information Technology         0.0231
Financials                     0.0207


In [22]:
# Start with 1 and compound returns
benchmark_cum = (1 + benchmark_sectors_daily_returns).cumprod()
optimised_cum = (1 + optimised_sectors_daily_returns).cumprod()

initial_investment = 100_000

benchmark_final_values = benchmark_cum.iloc[-1] * initial_investment
optimised_final_values = optimised_cum.iloc[-1] * initial_investment

difference = optimised_final_values - benchmark_final_values

final_df = pd.DataFrame({
    "Benchmark_Final": benchmark_final_values,
    "Optimised_Final": optimised_final_values,
    "Difference": difference
}).round(2)

print(final_df.sort_values("Difference", ascending=False))


                        Benchmark_Final  Optimised_Final  Difference
Information Technology         80085.21         80576.17      490.95
Financials                     84076.97         84161.12       84.15
Energy                         95531.76         95473.92      -57.84
Industrials                    86871.87         86663.12     -208.75
Materials                      85814.78         84589.55    -1225.23
Utilities                      95287.80         93734.83    -1552.96
Consumer Staples               95673.34         94041.45    -1631.88
Health Care                    95438.23         93119.79    -2318.44
Consumer Discretionary         75293.42         72869.63    -2423.79
Real Estate                    86334.82         83012.42    -3322.40
Communication Services         81669.78         76826.79    -4842.99


In [23]:
import pandas as pd

# --- Daily sector weights ---
sector_sp500_weights = float_mcap.groupby(sector_map_series, axis=1).sum()
sector_sp500_weights = sector_sp500_weights.div(sector_sp500_weights.sum(axis=1), axis=0)

# --- Cumulative growth per sector ---
benchmark_cum = (1 + benchmark_sectors_daily_returns).cumprod()
optimised_cum = (1 + optimised_sectors_daily_returns).cumprod()

# --- Start with $100k ---
initial_investment = 100_000

# Align
benchmark_cum, sector_sp500_weights = benchmark_cum.align(sector_sp500_weights, join="inner", axis=0)
optimised_cum, sector_sp500_weights = optimised_cum.align(sector_sp500_weights, join="inner", axis=0)

# --- Portfolio values (daily evolving) ---
benchmark_portfolio_value = (benchmark_cum * sector_sp500_weights).sum(axis=1)
optimised_portfolio_value = (optimised_cum * sector_sp500_weights).sum(axis=1)

# Scale both portfolios to same baseline (benchmark scaling)
scale_factor = initial_investment / benchmark_portfolio_value.iloc[0]
benchmark_portfolio_value *= scale_factor
optimised_portfolio_value *= scale_factor

# --- Final results ---
final_benchmark = benchmark_portfolio_value.iloc[-1]
final_optimised = optimised_portfolio_value.iloc[-1]
difference = final_optimised - final_benchmark


# --- Sector-level portfolio values ---
benchmark_sector_values = benchmark_cum * sector_sp500_weights * scale_factor
optimised_sector_values = optimised_cum * sector_sp500_weights * scale_factor

# --- Final sector contributions ---
final_benchmark_by_sector = benchmark_sector_values.iloc[-1]
final_optimised_by_sector = optimised_sector_values.iloc[-1]

sector_contributions = final_optimised_by_sector - final_benchmark_by_sector

# Contributions in $
sector_contributions_dollars = sector_contributions

# Contributions in % of total difference
sector_contributions_pct = sector_contributions_dollars / difference * 100

# Build results table
sector_results = pd.DataFrame({
    "Contribution ($)": sector_contributions_dollars,
    "Contribution (%)": sector_contributions_pct
})

# Add totals
sector_results.loc["Total"] = [difference, 100.0]                      # sector shares sum to 100%

# Format nicely
print(sector_results.round(2).sort_values("Contribution ($)", ascending=False))


portfolio_summary = pd.DataFrame({
    "Final value ($)": [final_benchmark, final_optimised],
    "Return (%)": [
        (final_benchmark / initial_investment - 1) * 100,
        (final_optimised / initial_investment - 1) * 100
    ]
}, index=["Benchmark", "Decarbonised"])

# Add row for difference
portfolio_summary.loc["Difference"] = [
    difference,
    (final_optimised / final_benchmark - 1) * 100
]

print(portfolio_summary.round(2))

                        Contribution ($)  Contribution (%)
Information Technology            113.68             -9.22
Financials                         11.09             -0.90
Energy                             -2.48              0.20
Industrials                       -17.57              1.42
Materials                         -32.08              2.60
Utilities                         -46.89              3.80
Real Estate                       -94.19              7.64
Consumer Staples                 -122.97              9.97
Consumer Discretionary           -250.99             20.35
Health Care                      -351.11             28.46
Communication Services           -440.16             35.68
Total                           -1233.67            100.00
              Final value ($)  Return (%)
Benchmark            85483.56      -14.52
Decarbonised         84249.89      -15.75
Difference           -1233.67       -1.44


/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/2660075039.py:4: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  sector_sp500_weights = float_mcap.groupby(sector_map_series, axis=1).sum()


## Rerunning TE estimation and comparing benchmark vs decarbonised portfolios, now including stocks (OGN, CARR, OTIS) that were excluded earlier in the benchmark because of missing return data in the covariance period

In [24]:
price_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='CLOSE PRICE', header = 4)
price_next_3m.columns = price_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
price_next_3m = price_next_3m.iloc[:-1]

div_rate_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='DIV RATE', header = 4)
div_rate_next_3m.columns = div_rate_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_rate_next_3m = div_rate_next_3m.iloc[:-1]

div_date_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='DIV DATE', header = 4)
div_date_next_3m.columns = div_date_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_date_next_3m = div_date_next_3m.iloc[:-1]

ffnosh_next_3m = pd.read_excel('Data/out-of-sample/price_div_comp_0322.xlsm', sheet_name='FFNOSH', header = 4)
ffnosh_next_3m.columns =ffnosh_next_3m.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
ffnosh_next_3m = ffnosh_next_3m.iloc[:-1]

div_date_next_3m.columns = price_next_3m.columns
div_rate_next_3m.columns = price_next_3m.columns
ffnosh_next_3m.columns = price_next_3m.columns

price_next_3m.index = price_next_3m.Code
div_rate_next_3m.index = div_rate_next_3m.Code
div_date_next_3m.index = div_date_next_3m.Code
ffnosh_next_3m.index = ffnosh_next_3m.Code

price_next_3m = price_next_3m.iloc[:, 1:]
div_rate = div_rate_next_3m.iloc[:, 1:]
div_date_next_3m = div_date_next_3m.iloc[:, 1:]
ffnosh_next_3m = ffnosh_next_3m.iloc[:, 1:]

type_lseg = pd.read_excel("Data/out-of-sample/symbol_comp_0322.xlsm", sheet_name="SYMBOL", dtype=str).iloc[0].values[1:]
symbol_lseg = pd.read_excel("Data/out-of-sample/symbol_comp_0322.xlsm", sheet_name="SYMBOL", dtype=str, header=2)
symbol_lseg = symbol_lseg.iloc[:1].transpose().reset_index().rename(columns= {"index": "NAME", 0: "SYMBOL"}).iloc[1:]
symbol_lseg['TYPE'] = type_lseg

symbol_type_matches  = symbol_lseg.set_index("TYPE")["SYMBOL"].to_dict()

price_next_3m = price_next_3m.rename(columns=symbol_type_matches)
ffnosh_next_3m = ffnosh_next_3m.rename(columns=symbol_type_matches)


# change symbol from BRK.A to BRK-B:
price_next_3m.rename(columns={'BRK.A':'BRK-B'}, inplace=True)
ffnosh_next_3m.rename(columns={'BRK.A':'BRK-B'}, inplace=True)

price_next_3m.index = pd.to_datetime(price_next_3m.index)
ffnosh_next_3m.index = pd.to_datetime(ffnosh_next_3m.index)

# Example duplicate map
duplicate_map = {
    "GOOG": "GOOGL",  # aggregate under GOOG
    "FOX": "FOXA",    # aggregate under FOXA
    "NWS": "NWSA"     # aggregate under NWSA
}

# --- Step 1: Align indexes ---
price_next_3m.index = pd.to_datetime(price_next_3m.index)
ffnosh_next_3m.index = pd.to_datetime(ffnosh_next_3m.index)

# --- Step 2: Apply duplicate mapping ---
price_next_3m = price_next_3m.rename(columns=duplicate_map)
ffnosh_next_3m = ffnosh_next_3m.rename(columns=duplicate_map)

# After renaming, duplicates will exist → group and aggregate
def aggregate_duplicates(price_df, ffnosh_df):
    # Weighted price = Σ(price * shares) / Σ(shares)
    weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
    total_ffnosh = ffnosh_df.groupby(axis=1, level=0).sum()
    return weighted_price, total_ffnosh


price_agg, ffnosh_agg = aggregate_duplicates(price_next_3m, ffnosh_next_3m)

# --- Step 3: Float-adjusted market cap ---
float_mcap = price_agg * ffnosh_agg

data = pd.read_excel("Data/out-of-sample/dataset_comp_0322.xlsx")
# --- Step 4: Add sector info ---
sector_map = data.set_index("SYMBOL")["GICS Sector"].to_dict()

sector_map.update({'OGN':'Health Care', 'CEG':'Utilities'})

sector_map_series = pd.Series(sector_map)

# Columns in float_mcap but not in sector_map_series
extra_in_float_mcap = float_mcap.columns.difference(sector_map_series.index)

# Symbols in sector_map_series but not in float_mcap
extra_in_sector_map = sector_map_series.index.difference(float_mcap.columns)

print("In float_mcap but not in sector_map_series:", extra_in_float_mcap.tolist())
print("In sector_map_series but not in float_mcap:", extra_in_sector_map.tolist())

# Make sure columns align
#float_mcap = float_mcap.loc[:, sector_map_series.index]

# --- Step 5: Compute sector-level weights daily ---
sector_weights = {}
for sector in sector_map_series.unique():
    tickers = sector_map_series[sector_map_series == sector].index
    sector_mcap = float_mcap[tickers].sum(axis=1)
    sector_weights[sector] = float_mcap[tickers].div(sector_mcap, axis=0)

sector_portfolio_returns = {}

for sector, weights_df in sector_weights.items():
    # Align weights with returns (common tickers + dates)
    common_tickers = weights_df.columns.intersection(daily_returns_next_3m.columns)
    common_index = weights_df.index.intersection(daily_returns_next_3m.index)

    w = weights_df.loc[common_index, common_tickers]
    r = daily_returns_next_3m.loc[common_index, common_tickers]

    # Multiply elementwise and sum across tickers → portfolio return per day
    sector_returns = (w * r).sum(axis=1)

    sector_portfolio_returns[sector] = sector_returns

# Put all in a single DataFrame
sector_portfolio_df = pd.DataFrame(sector_portfolio_returns)


# --- Daily sector weights ---
sector_sp500_weights = float_mcap.groupby(sector_map_series, axis=1).sum()
sector_sp500_weights = sector_sp500_weights.div(sector_sp500_weights.sum(axis=1), axis=0)

# --- Cumulative growth per sector ---
benchmark_cum = (1 + benchmark_sectors_daily_returns).cumprod()
optimised_cum = (1 + optimised_sectors_daily_returns).cumprod()

# --- Start with $100k ---
initial_investment = 100_000

# Align
benchmark_cum, sector_sp500_weights = benchmark_cum.align(sector_sp500_weights, join="inner", axis=0)
optimised_cum, sector_sp500_weights = optimised_cum.align(sector_sp500_weights, join="inner", axis=0)

# --- Portfolio values (daily evolving) ---
benchmark_portfolio_value = (benchmark_cum * sector_sp500_weights).sum(axis=1)
optimised_portfolio_value = (optimised_cum * sector_sp500_weights).sum(axis=1)

# Scale both portfolios to same baseline (benchmark scaling)
scale_factor = initial_investment / benchmark_portfolio_value.iloc[0]
benchmark_portfolio_value *= scale_factor
optimised_portfolio_value *= scale_factor

# --- Final results ---
final_benchmark = benchmark_portfolio_value.iloc[-1]
final_optimised = optimised_portfolio_value.iloc[-1]
difference = final_optimised - final_benchmark


# --- Sector-level portfolio values ---
benchmark_sector_values = benchmark_cum * sector_sp500_weights * scale_factor
optimised_sector_values = optimised_cum * sector_sp500_weights * scale_factor

# --- Final sector contributions ---
final_benchmark_by_sector = benchmark_sector_values.iloc[-1]
final_optimised_by_sector = optimised_sector_values.iloc[-1]

sector_contributions = final_optimised_by_sector - final_benchmark_by_sector

# Contributions in $
sector_contributions_dollars = sector_contributions

# Contributions in % of total difference
sector_contributions_pct = sector_contributions_dollars / difference * 100

# Build results table
sector_results = pd.DataFrame({
    "Contribution ($)": sector_contributions_dollars,
    "Contribution (%)": sector_contributions_pct
})

# Add totals
sector_results.loc["Total"] = [difference, 100.0]                      # sector shares sum to 100%

# Format nicely
print(sector_results.round(2).sort_values("Contribution ($)", ascending=False))


portfolio_summary = pd.DataFrame({
    "Final value ($)": [final_benchmark, final_optimised],
    "Return (%)": [
        (final_benchmark / initial_investment - 1) * 100,
        (final_optimised / initial_investment - 1) * 100
    ]
}, index=["Benchmark", "Decarbonised"])

# Add row for difference
portfolio_summary.loc["Difference"] = [
    difference,
    (final_optimised / final_benchmark - 1) * 100
]

print(portfolio_summary.round(2))
sector_results.to_excel("Data/te-testing-results/benchmark_vs_decarb_sectoral_contribution_proportional_0322.xlsx")
portfolio_summary.to_excel("Data/te-testing-results/benchmark_vs_decarb_final_value_proportional_0322.xlsx")

In float_mcap but not in sector_map_series: []
In sector_map_series but not in float_mcap: []
                        Contribution ($)  Contribution (%)
Information Technology            113.59             -9.20
Financials                         11.08             -0.90
Energy                             -2.48              0.20
Industrials                       -17.56              1.42
Materials                         -32.05              2.60
Utilities                         -47.65              3.86
Real Estate                       -94.12              7.63
Consumer Staples                 -122.87              9.96
Consumer Discretionary           -250.79             20.32
Health Care                      -351.49             28.48
Communication Services           -439.80             35.64
Total                           -1234.13            100.00
              Final value ($)  Return (%)
Benchmark            85490.67      -14.51
Decarbonised         84256.54      -15.74
Difference   

/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/1156177684.py:67: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/1156177684.py:67: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  weighted_price = (price_df * ffnosh_df).groupby(axis=1, level=0).sum() / ffnosh_df.groupby(axis=1, level=0).sum()
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/1156177684.py:68: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  total_ffnosh = ffnosh_df.groupby(axis=1, level=0).sum()
/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_39474/1156177684.py:124: FutureWarning: DataFrame.groupby with axis=1 is depre

In [25]:
benchmark_sectors_daily_returns = sector_portfolio_df
optimised_sectors_daily_returns = pd.DataFrame(decarb_sector_returns)

In [26]:
import numpy as np
import pandas as pd

sector_te = {}

for sector in benchmark_sectors_daily_returns.columns:
    # Align indices
    r_b = benchmark_sectors_daily_returns[sector]
    r_d = optimised_sectors_daily_returns[sector]
    common_idx = r_b.index.intersection(r_d.index)
    r_b, r_d = r_b.loc[common_idx], r_d.loc[common_idx]

    # Active returns
    active = r_d - r_b

    # Daily TE → annualised
    te_daily = active.std()
    te_ann = te_daily * np.sqrt(252)

    sector_te[sector] = te_ann

# Build DataFrame
te_df = pd.DataFrame.from_dict(sector_te, orient="index", columns=["Annualised_TE"])
print(te_df.sort_values("Annualised_TE", ascending=False).round(4))
te_df.to_excel("Data/te-testing-results/te_0322.xlsx")

                        Annualised_TE
Real Estate                    0.0530
Communication Services         0.0466
Consumer Discretionary         0.0371
Industrials                    0.0309
Consumer Staples               0.0291
Health Care                    0.0290
Energy                         0.0276
Materials                      0.0269
Utilities                      0.0259
Information Technology         0.0231
Financials                     0.0207


In [27]:
# Start with 1 and compound returns
benchmark_cum = (1 + benchmark_sectors_daily_returns).cumprod()
optimised_cum = (1 + optimised_sectors_daily_returns).cumprod()

initial_investment = 100_000

# Final values by sector
benchmark_final_values = benchmark_cum.iloc[-1] * initial_investment
optimised_final_values = optimised_cum.iloc[-1] * initial_investment

# Differences in final $ values
difference = optimised_final_values - benchmark_final_values

# Returns relative to initial investment
benchmark_returns = (benchmark_final_values / initial_investment - 1) * 100
optimised_returns = (optimised_final_values / initial_investment - 1) * 100
difference_returns = optimised_returns - benchmark_returns

# Build DataFrame
final_df = pd.DataFrame({
    "Benchmark_Final": benchmark_final_values,
    "Optimised_Final": optimised_final_values,
    "Difference ($)": difference,
    "Benchmark_Return (%)": benchmark_returns,
    "Optimised_Return (%)": optimised_returns,
    "Difference_Return (%)": difference_returns
}).round(2)

print(final_df.sort_values("Difference ($)", ascending=False))
final_df.to_excel("Data/te-testing-results/benchmark_vs_decarb_final_value_equal_weight_0322.xlsx")

                        Benchmark_Final  Optimised_Final  Difference ($)  \
Information Technology         80085.21         80576.17          490.95   
Financials                     84076.97         84161.12           84.15   
Energy                         95531.76         95473.92          -57.84   
Industrials                    86871.87         86663.12         -208.75   
Materials                      85814.78         84589.55        -1225.23   
Consumer Staples               95673.34         94041.45        -1631.88   
Utilities                      95376.29         93734.83        -1641.46   
Health Care                    95447.61         93119.79        -2327.81   
Consumer Discretionary         75293.42         72869.63        -2423.79   
Real Estate                    86334.82         83012.42        -3322.40   
Communication Services         81669.78         76826.79        -4842.99   

                        Benchmark_Return (%)  Optimised_Return (%)  \
Information Techn